In [1]:
!pip install -q transformers accelerate bitsandbytes h5py pandas pyarrow torchvision qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 36.0 MB/s eta 0:00:00


In [2]:
import os
import io
import gc
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import h5py
import numpy as np
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from transformers import T5Tokenizer, T5EncoderModel
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# ==========================================
# 0. DATALOADER (ĐÃ TEST THÀNH CÔNG: DIM = 26)
# ==========================================
class RealRobotDataset(Dataset):
    def __init__(self, file_path, chunk_size=10, image_size=(224, 224), max_samples=None):
        self.chunk_size = chunk_size
        self.transform = T.Compose([T.Resize(image_size), T.ToTensor()])
        self.data = []
        
        if file_path.endswith('.parquet'): self._load_parquet(file_path, max_samples)
        elif file_path.endswith('.hdf5'): self._load_hdf5(file_path, max_samples)
        
        if len(self.data) > 0:
            self.state_dim = self.data[0][1].shape[0]
            self.action_dim = self.data[0][3].shape[1]

    def _load_parquet(self, path, max_samples):
        df = pd.read_parquet(path)
        count = 0
        for episode_id, episode_df in df.groupby('episode_index'):
            episode_df = episode_df.reset_index(drop=True)
            for t in range(len(episode_df) - self.chunk_size):
                if max_samples and count >= max_samples: return
                row = episode_df.iloc[t]
                
                img_dict = row['observation.images.head_camera']
                image = Image.open(io.BytesIO(img_dict['bytes'])).convert("RGB") if isinstance(img_dict, dict) and 'bytes' in img_dict else Image.fromarray(np.array(img_dict, dtype=np.uint8)).convert("RGB")
                img_tensor = self.transform(image)
                
                state_tensor = torch.tensor(row['observation.state'], dtype=torch.float32)
                text = f"Perform task {row['task_index']}" 
                action_tensor = torch.tensor(episode_df.iloc[t : t + self.chunk_size]['action'].tolist(), dtype=torch.float32)
                
                self.data.append((img_tensor, state_tensor, text, action_tensor))
                count += 1

    def _load_hdf5(self, path):
        print(f"Đang xử lý Mimicgen HDF5: {path}...")
        try:
            with h5py.File(path, 'r') as f:
                demos = list(f['data'].keys()) # ['demo_0', 'demo_1', ...]
                
                for demo_name in demos[:50]: # Lấy 50 episode đầu tiên để làm Seed
                    demo = f['data'][demo_name]
                    
                    # Tìm key chứa hình ảnh và state bên trong 'obs'
                    obs = demo['obs']
                    img_key = [k for k in obs.keys() if 'image' in k or 'rgb' in k][0]
                    state_key = [k for k in obs.keys() if 'state' in k or 'pos' in k or 'qpos' in k][0]
                    
                    images = obs[img_key][:]
                    states = obs[state_key][:]
                    actions = demo['actions'][:]
                    
                    T_steps = len(actions)
                    
                    for t in range(T_steps - self.chunk_size):
                        # Ảnh dạng Numpy (H, W, C) -> PIL Image -> Tensor [3, 224, 224]
                        img_arr = images[t]
                        if img_arr.shape[0] == 3: # Xử lý nếu ảnh bị ngược kênh (C, H, W)
                            img_arr = img_arr.transpose(1, 2, 0)
                        image = Image.fromarray(img_arr).convert("RGB")
                        img_tensor = self.transform(image)
                        
                        state_tensor = torch.tensor(states[t], dtype=torch.float32)
                        
                        action_chunk = actions[t : t + self.chunk_size]
                        action_tensor = torch.tensor(action_chunk, dtype=torch.float32)
                        
                        self.data.append((img_tensor, state_tensor, "Mimic this trajectory", action_tensor))
                        
        except Exception as e:
            print(f"Có lỗi khi xử lý HDF5: {e}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

🚀 Using device: cuda


In [3]:
# ==========================================
# 1. CORE MODELS: SUPERTINY & TARGET
# ==========================================
class SuperTinyVLA(nn.Module):
    def __init__(self, action_dim, state_dim, hidden_dim=256, chunk_size=10):
        super().__init__()
        self.image_encoder = nn.Sequential(*list(models.resnet18(pretrained=True).children())[:-1])
        self.image_proj = nn.Linear(512, hidden_dim) 
        self.tokenizer = T5Tokenizer.from_pretrained("t5-small", legacy=False)
        self.text_encoder = T5EncoderModel.from_pretrained("t5-small")
        for p in self.text_encoder.parameters(): p.requires_grad = False
        self.text_proj = nn.Linear(512, hidden_dim) 
        self.state_encoder = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU(), nn.Linear(128, hidden_dim))
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=4)
        self.action_queries = nn.Parameter(torch.randn(1, chunk_size, hidden_dim))
        self.action_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, images, states, texts):
        B = images.size(0)
        img_tok = self.image_proj(self.image_encoder(images).view(B, -1)).unsqueeze(1)
        state_tok = self.state_encoder(states).unsqueeze(1)
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(images.device)
        text_tok = self.text_proj(self.text_encoder(**inputs).last_hidden_state)
        
        memory = torch.cat([img_tok, state_tok, text_tok], dim=1)
        queries = self.action_queries.expand(B, -1, -1)
        return self.action_head(self.transformer_decoder(tgt=queries, memory=memory))

class TargetSmolVLA(nn.Module):
    def __init__(self, action_dim, state_dim, hidden_dim=256, chunk_size=10):
        super().__init__()
        self.feature_extractor = SuperTinyVLA(action_dim, state_dim, hidden_dim, chunk_size)
        self.action_proj = nn.Linear(action_dim, hidden_dim) 
        expert_layer = nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.action_expert = nn.TransformerDecoder(expert_layer, num_layers=4)
        self.vector_field_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, images, states, texts, noisy_actions):
        B = images.size(0)
        img_tok = self.feature_extractor.image_proj(self.feature_extractor.image_encoder(images).view(B, -1)).unsqueeze(1)
        state_tok = self.feature_extractor.state_encoder(states).unsqueeze(1)
        inputs = self.feature_extractor.tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(images.device)
        text_tok = self.feature_extractor.text_proj(self.feature_extractor.text_encoder(**inputs).last_hidden_state)
        
        obs_memory = torch.cat([img_tok, state_tok, text_tok], dim=1) 
        action_tokens = self.action_proj(noisy_actions) 
        expert_out = self.action_expert(tgt=action_tokens, memory=obs_memory)
        return self.vector_field_head(expert_out)

In [4]:
# ==========================================
# 2. VLM EVALUATOR (QWEN-VL 2B THEO YÊU CẦU)
# ==========================================
class QwenVLEvaluator:
    def __init__(self, device, threshold=7.0):
        self.device = device
        self.threshold = threshold
        print("🤖 Đang tải Qwen-VL 2B (Teacher Evaluator)...")
        # Load mô hình ở bfloat16 để tiết kiệm RAM trên Kaggle
        model_id = "Qwen/Qwen2-VL-2B-Instruct"
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="auto"
        )
        self.model.eval()

    def evaluate(self, image_tensor, text, gen_action, true_action):
        # 1. Chuyển Tensor về dạng PIL Image cho Qwen
        img_np = (image_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        pil_img = Image.fromarray(img_np)
        
        # 2. Prompt cho Qwen-VL (Chấm điểm bối cảnh 1-10)
        prompt = f"Look at this robot camera view. The task is '{text}'. Is the environment suitable for this task? Rate the visual context feasibility from 1 to 10. Just reply with the number."
        
        messages = [{"role": "user", "content": [{"type": "image", "image": pil_img}, {"type": "text", "text": prompt}]}]
        text_input = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = self.processor(text=[text_input], images=[pil_img], padding=True, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=10)
            output_text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        # Parse điểm số từ Qwen (Lấy con số đầu tiên tìm thấy)
        match = re.search(r'\b(10|[1-9])\b', output_text)
        qwen_score = float(match.group(0)) if match else 5.0 # Mặc định 5 nếu không parse được
        
        # 3. Tính Action MSE
        mse_error = torch.mean((gen_action - true_action)**2).item()
        action_score = max(0.0, 10.0 - (mse_error * 20.0))
        
        # 4. Gộp điểm (30% Qwen Vision Context + 70% Action Error)
        final_score = (0.3 * qwen_score) + (0.7 * action_score)
        return final_score >= self.threshold

In [5]:
# ==========================================
# 3. PHẦN 1: TRAINING TỪ ĐẦU & BOOTSTRAPPING
# ==========================================
def run_part1(file_path):
    print("\n" + "="*50)
    print("🌟 PHẦN 1: INITIALIZATION & SEED-TO-SCALE")
    print("="*50)
    
    # 1. Load Data
    dataset = RealRobotDataset(file_path, max_samples=64) # Giới hạn 64 sample để train nhanh trên Kaggle
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
    A_DIM, S_DIM = dataset.action_dim, dataset.state_dim
    
    # 2. Train SuperTiny (Worker)
    print("\n--- 1. WARM-UP WORKER MODEL ---")
    worker = SuperTinyVLA(A_DIM, S_DIM).to(device)
    optimizer = torch.optim.Adam(worker.parameters(), lr=1e-4)
    worker.train()
    for epoch in range(1): # 1 Epoch theo yêu cầu
        for imgs, states, texts, actions in dataloader:
            imgs, states, actions = imgs.to(device), states.to(device), actions.to(device)
            loss = nn.MSELoss()(worker(imgs, states, texts), actions)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        print(f"Worker Epoch 1 Loss: {loss.item():.4f}")

    # 3. VLM Evaluator (Sinh dữ liệu & Lọc)
    print("\n--- 2. QWEN-VL BOOTSTRAPPING ---")
    worker.eval()
    qwen_judge = QwenVLEvaluator(device)
    silver_data = []
    
    with torch.no_grad():
        for imgs, states, texts, true_actions in dataloader:
            imgs_gpu, states_gpu = imgs.to(device), states.to(device)
            gen_actions = worker(imgs_gpu, states_gpu, texts)
            
            for i in range(imgs.size(0)):
                passed = qwen_judge.evaluate(imgs[i], texts[i], gen_actions[i].cpu(), true_actions[i])
                if passed:
                    silver_data.append((imgs[i].cpu(), states[i].cpu(), texts[i], gen_actions[i].cpu()))
    
    print(f"✅ Qwen-VL đã lọc được {len(silver_data)} trajectory xịn vào Silver Dataset.")
    
    # Kỹ thuật Kaggle: Xóa Qwen và Worker khỏi RAM để nhường chỗ cho Target Model
    del qwen_judge; del worker; gc.collect(); torch.cuda.empty_cache()

    # 4. Train Target Model (Flow Matching)
    print("\n--- 3. TRAIN TARGET MODEL (FLOW MATCHING) ---")
    full_dataset = dataset.data + silver_data
    scale_loader = DataLoader(full_dataset, batch_size=8, shuffle=True)
    
    target_model = TargetSmolVLA(A_DIM, S_DIM).to(device)
    optimizer_target = torch.optim.Adam(target_model.parameters(), lr=1e-4)
    
    target_model.train()
    for epoch in range(100):
        for imgs, states, texts, true_actions in scale_loader:
            imgs, states, true_actions = imgs.to(device), states.to(device), true_actions.to(device)
            B = true_actions.size(0)
            sigma = torch.rand(B, 1, 1).to(device)
            noise = torch.randn_like(true_actions).to(device)
            
            # Flow Matching Equation
            noisy_actions = sigma * true_actions + (1 - sigma) * noise
            target_vf = true_actions - noise
            
            pred_vf = target_model(imgs, states, texts, noisy_actions)
            loss = nn.MSELoss()(pred_vf, target_vf)
            
            optimizer_target.zero_grad(); loss.backward(); optimizer_target.step()
        print(f"Target Model Epoch {epoch+1} Loss L_FM: {loss.item():.4f}")

    # 5. Lưu Pretrained Model
    torch.save(target_model.state_dict(), "seed2scale_part1_pretrained.pth")
    print("💾 Đã lưu 'seed2scale_part1_pretrained.pth'")
    return A_DIM, S_DIM

In [6]:
# ==========================================
# 4. PHẦN 2: FINE-TUNING TRÊN NOVEL DATASET
# ==========================================
def run_part2(file_path_novel, A_DIM, S_DIM):
    print("\n" + "="*50)
    print("🔥 PHẦN 2: FINE-TUNING PRE-TRAINED MODEL TRÊN NOVEL DATASET")
    print("="*50)
    
    # 1. Load Novel Dataset (Tập dữ liệu mới)
    print("Đang nạp Novel Dataset...")
    novel_dataset = RealRobotDataset(file_path_novel, max_samples=64)
    novel_loader = DataLoader(novel_dataset, batch_size=8, shuffle=True)
    
    # 2. Khởi tạo Model và Load Trọng số từ Phần 1
    model_finetune = TargetSmolVLA(A_DIM, S_DIM).to(device)
    model_finetune.load_state_dict(torch.load("seed2scale_part1_pretrained.pth", map_location=device))
    print("✅ Đã load thành công Pretrained Weights từ Phần 1.")
    
    # 3. Fine-tuning (Cập nhật trọng số của Decoder & Action Expert)
    # Đóng băng Encoder ảnh/text để quá trình Fine-tune nhanh và tránh quên (Catastrophic Forgetting)
    for param in model_finetune.feature_extractor.image_encoder.parameters(): param.requires_grad = False
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model_finetune.parameters()), lr=5e-5) # Learning rate nhỏ lại
    
    model_finetune.train()
    for epoch in range(100): # Fine-tune ít nhất 1-2 epoch
        total_loss = 0
        for imgs, states, texts, true_actions in novel_loader:
            imgs, states, true_actions = imgs.to(device), states.to(device), true_actions.to(device)
            B = true_actions.size(0)
            sigma = torch.rand(B, 1, 1).to(device)
            noise = torch.randn_like(true_actions).to(device)
            
            noisy_actions = sigma * true_actions + (1 - sigma) * noise
            target_vf = true_actions - noise
            
            pred_vf = model_finetune(imgs, states, texts, noisy_actions)
            loss = nn.MSELoss()(pred_vf, target_vf)
            
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            
        print(f"Fine-Tuning Epoch {epoch+1} | Novel Dataset Loss: {total_loss/len(novel_loader):.4f}")

    # 4. Lưu Model Fine-tuned
    torch.save(model_finetune.state_dict(), "seed2scale_part2_finetuned.pth")
    print("💾 Đã lưu model hoàn chỉnh 'seed2scale_part2_finetuned.pth'. Hoàn thành Lab!")

In [7]:
# ==========================================
# 5. CHẠY CHÍNH (MAIN ENTRY)
# ==========================================
if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore")
    
    SEED_DATA = "/kaggle/input/datasets/tranbao0105/ml-lab1-realmirror/RealMirror.parquet"
    NOVEL_DATA = "/kaggle/input/datasets/tranbao0105/ml-lab1-realmirror/RealMirror.parquet"
    
    # Chạy Phần 1
    a_dim, s_dim = run_part1(SEED_DATA)
    
    # Chạy Phần 2
    run_part2(NOVEL_DATA, a_dim, s_dim)


🌟 PHẦN 1: INITIALIZATION & SEED-TO-SCALE

--- 1. WARM-UP WORKER MODEL ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

✅ Qwen-VL đã lọc được 0 trajectory xịn vào Silver Dataset.

--- 3. TRAIN TARGET MODEL (FLOW MATCHING) ---


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

Target Model Epoch 1 Loss L_FM: 110274.7969
Target Model Epoch 2 Loss L_FM: 108853.4922
Target Model Epoch 3 Loss L_FM: 108709.1719
Target Model Epoch 4 Loss L_FM: 108675.4766
Target Model Epoch 5 Loss L_FM: 113576.6406
Target Model Epoch 6 Loss L_FM: 116216.1406
Target Model Epoch 7 Loss L_FM: 116186.5078
Target Model Epoch 8 Loss L_FM: 108531.7266
Target Model Epoch 9 Loss L_FM: 108506.6641
Target Model Epoch 10 Loss L_FM: 112275.1562
Target Model Epoch 11 Loss L_FM: 109503.4297
Target Model Epoch 12 Loss L_FM: 108379.6406
Target Model Epoch 13 Loss L_FM: 108331.7969
Target Model Epoch 14 Loss L_FM: 108308.9297
Target Model Epoch 15 Loss L_FM: 108269.4141
Target Model Epoch 16 Loss L_FM: 108218.6484
Target Model Epoch 17 Loss L_FM: 119624.9375
Target Model Epoch 18 Loss L_FM: 119578.2734
Target Model Epoch 19 Loss L_FM: 111945.5625
Target Model Epoch 20 Loss L_FM: 108092.4453
Target Model Epoch 21 Loss L_FM: 108047.2656
Target Model Epoch 22 Loss L_FM: 108013.3203
Target Model Epoch 

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

✅ Đã load thành công Pretrained Weights từ Phần 1.
Fine-Tuning Epoch 1 | Novel Dataset Loss: 105338.1768
Fine-Tuning Epoch 2 | Novel Dataset Loss: 105295.3770
Fine-Tuning Epoch 3 | Novel Dataset Loss: 105259.2676
Fine-Tuning Epoch 4 | Novel Dataset Loss: 105231.2314
Fine-Tuning Epoch 5 | Novel Dataset Loss: 105212.0947
Fine-Tuning Epoch 6 | Novel Dataset Loss: 105189.6250
Fine-Tuning Epoch 7 | Novel Dataset Loss: 105144.8525
Fine-Tuning Epoch 8 | Novel Dataset Loss: 105124.6914
Fine-Tuning Epoch 9 | Novel Dataset Loss: 105089.8008
Fine-Tuning Epoch 10 | Novel Dataset Loss: 105066.9424
Fine-Tuning Epoch 11 | Novel Dataset Loss: 105042.6738
Fine-Tuning Epoch 12 | Novel Dataset Loss: 105021.7578
Fine-Tuning Epoch 13 | Novel Dataset Loss: 104985.0703
Fine-Tuning Epoch 14 | Novel Dataset Loss: 104963.7412
Fine-Tuning Epoch 15 | Novel Dataset Loss: 104939.6797
Fine-Tuning Epoch 16 | Novel Dataset Loss: 104909.7871
Fine-Tuning Epoch 17 | Novel Dataset Loss: 104889.1416
Fine-Tuning Epoch 18 | 